# Trích xuất Loại Đất từ Quy Hoạch HCMC

Pipeline:
1. Đọc CSV crawl được (có cột `Latitude`, `Longitude` theo WGS84 từ Google Maps)
2. Chuyển đổi sang hệ tọa độ **VN2000** (Transverse Mercator, múi 48, kinh tuyến trục 105°)
3. Gọi API `thongtinquyhoach.hochiminhcity.gov.vn` để lấy loại đất tại từng toạ độ
4. Ghi kết quả ra CSV mới

# Phát hiện vấn đề một toạ độ trả về thông tin của một ô đất chứa nó. Thông tin đó bao gồm nhiều loại đất quy hoạch và diện tích theo loại. 

# Crawl đầy đủ thông tin hơn.

In [ ]:
import ast
import csv
import json
import time
import requests

# ==========================================
# CẤU HÌNH TẠI ĐÂY
# ==========================================
FILE_INPUT       = "/Users/doanlong/Documents/Môn học/Thu thập và Tiền xử lý dữ liệu/house_price_expanded/Bản_ghi_chỉ_có_trong_bds.csv"
FILE_OUTPUT      = "/Users/doanlong/Documents/Môn học/Thu thập và Tiền xử lý dữ liệu/house_price_expanded/Bản_ghi_chỉ_có_trong_bds_quyhoach.csv"
MAX_RETRY        = 3
BACKOFF_FACTOR   = 3
DELAY_GIUA_DONG  = 1        # giây nghỉ giữa các dòng
OFFSET_DEGREE    = 0.00005  # ~5.5m — dùng khi điểm gốc chỉ có đất giao thông
# ==========================================

session = requests.Session()
session.headers.update({
    "Host":         "sqhkt-qlqh.tphcm.gov.vn",
    "Accept":       "application/json, text/plain, */*",
    "Content-Type": "application/x-www-form-urlencoded",
    "User-Agent":   "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Origin":       "https://thongtinquyhoach.hochiminhcity.gov.vn",
    "Referer":      "https://thongtinquyhoach.hochiminhcity.gov.vn/"
})

# ==============================================================
# CẤU TRÚC CÁC CỘT ĐẦU RA
#
# Với lô đất có nhiều ô quy hoạch, mỗi trường QHPK gộp
# tất cả các giá trị lại bằng dấu " | " theo thứ tự ô.
#
# Ví dụ lô có 2 ô:
#   QHPK_Chuc_Nang = "Đất giao thông | Đất nhóm ở hiện trạng..."
#   QHPK_Dien_Tich = "1759.32 | 1566.05"
#   QHPK_Ty_Le     = "48.87% | 43.50%"
#   QHPK_Ma_QU     = "DGT | NNO"
#   QHPK_Ma_O_Pho  = " | II-12"
#   QHPK_So_O      = 2
# ==============================================================
CAC_COT_MOI = [
    "Quan_Huyen",
    "Phuong_Xa",
    "So_Thua",
    "So_To",
    "Dien_Tich_Lo",
    "Ten_Do_An",
    "QHPK_So_O",        # Số ô quy hoạch trả về
    "QHPK_Chuc_Nang",   # Tên chức năng, gộp bằng " | "
    "QHPK_Dien_Tich",   # Diện tích từng ô (m²), gộp bằng " | "
    "QHPK_Ty_Le",       # % diện tích từng ô, gộp bằng " | "
    "QHPK_Ma_QU",       # Mã quy ước (maquyuoc), gộp bằng " | "
    "QHPK_Ma_O_Pho",    # Mã ô phố (maopho), gộp bằng " | "
    "Trang_Thai",
]

SEP = " | "  # Dấu phân cách giữa các ô trong cùng 1 cột


# ──────────────────────────────────────────────────────────────
def _goi_api_mot_diem(lat, lon):
    """
    Gọi API cho 1 tọa độ. Trả về (result_data, None) nếu thành công,
    hoặc (None, "thông báo lỗi") nếu thất bại.
    """
    url  = "https://sqhkt-qlqh.tphcm.gov.vn/computing/930/api/v3.1/a-z/all"
    data = {"Lat": lat, "Lon": lon}

    for attempt in range(1, MAX_RETRY + 1):
        try:
            if attempt > 1:
                time.sleep(BACKOFF_FACTOR * (attempt - 1))

            response = session.post(url, data=data, timeout=15)

            if response.status_code != 200:
                return None, f"Lỗi HTTP {response.status_code}"

            raw_text = response.text.strip() if response.text else ""
            if not raw_text:
                return None, "Server trả về rỗng"

            # Parse JSON — fallback sang ast nếu server dùng nháy đơn
            try:
                result_data = response.json()
            except (json.JSONDecodeError, ValueError):
                try:
                    result_data = ast.literal_eval(raw_text)
                except Exception:
                    return None, f"Lỗi cấu trúc dữ liệu: {raw_text[:50]}"

            if isinstance(result_data, dict) and "error" in result_data:
                return None, f"Bị khóa: {result_data['error']}"
            if result_data.get("blocked") == 1:
                return None, "Bị chặn IP"

            return result_data, None  # ✅ thành công

        except requests.exceptions.Timeout:
            if attempt == MAX_RETRY:
                return None, "Mạng Timeout (Hết lượt thử lại)"
        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRY:
                return None, f"Lỗi mạng: {str(e)[:50]}"
        except Exception as e:
            return None, f"Lỗi không xác định: {str(e)[:50]}"

    return None, "Hết lượt retry"


def _boc_tach_ket_qua(result_data):
    """
    Chuyển dict raw từ API → dict kết quả theo CAC_COT_MOI.
    Các trường QHPK gộp nhiều ô bằng SEP = ' | '.
    """
    kq = {c: "" for c in CAC_COT_MOI}
    kq["QHPK_So_O"] = 0
    kq["Trang_Thai"] = "Thành công"

    # ── 1. ThongTinChung ────────────────────────────────────
    raw_chung = result_data.get("ThongTinChung", "")
    if raw_chung and raw_chung not in ("[]", "{}"):
        try:
            ttc = json.loads(raw_chung) if isinstance(raw_chung, str) else raw_chung
            if isinstance(ttc, dict):
                kq["Quan_Huyen"]   = ttc.get("tenquanhuyen", "")
                kq["Phuong_Xa"]    = ttc.get("tenphuongxa", "")
                kq["So_Thua"]      = ttc.get("sothua", "")
                kq["So_To"]        = ttc.get("soto", "")
                kq["Dien_Tich_Lo"] = ttc.get("dientich", "")
                dsdoan = ttc.get("dsdoan", [])
                if isinstance(dsdoan, list):
                    kq["Ten_Do_An"] = SEP.join(dsdoan)
        except Exception:
            kq["Trang_Thai"] = "Lỗi bóc tách ThongTinChung"

    # ── 2. QHPK — lấy TẤT CẢ ô, gộp thành 1 chuỗi ─────────
    raw_qhpk = result_data.get("QHPK", "")
    if raw_qhpk and raw_qhpk not in ("[]", "{}"):
        try:
            qhpk_list = json.loads(raw_qhpk) if isinstance(raw_qhpk, str) else raw_qhpk

            # Chuẩn hoá về list dù server trả object đơn
            if isinstance(qhpk_list, dict):
                qhpk_list = [qhpk_list]

            if isinstance(qhpk_list, list) and len(qhpk_list) > 0:
                ds_chuc_nang = []
                ds_dien_tich = []
                ds_ty_le     = []
                ds_ma_qu     = []
                ds_ma_o_pho  = []

                for item in qhpk_list:
                    props = item.get("properties", {}) if isinstance(item, dict) else {}

                    chuc_nang = props.get("chucnang") or props.get("chucNang") or ""
                    dien_tich = props.get("dientich", "")
                    ty_le     = props.get("tldientich", "")
                    ma_qu     = props.get("maquyuoc", "") or ""
                    ma_o_pho  = props.get("maopho", "")   or ""

                    # Làm tròn 2 chữ số thập phân
                    try:
                        dien_tich = f"{float(dien_tich):.2f}"
                    except (TypeError, ValueError):
                        dien_tich = str(dien_tich)

                    try:
                        ty_le = f"{float(ty_le):.2f}%"
                    except (TypeError, ValueError):
                        ty_le = str(ty_le)

                    ds_chuc_nang.append(chuc_nang)
                    ds_dien_tich.append(dien_tich)
                    ds_ty_le.append(ty_le)
                    ds_ma_qu.append(str(ma_qu))
                    ds_ma_o_pho.append(str(ma_o_pho))

                kq["QHPK_So_O"]      = len(qhpk_list)
                kq["QHPK_Chuc_Nang"] = SEP.join(ds_chuc_nang)
                kq["QHPK_Dien_Tich"] = SEP.join(ds_dien_tich)
                kq["QHPK_Ty_Le"]     = SEP.join(ds_ty_le)
                kq["QHPK_Ma_QU"]     = SEP.join(ds_ma_qu)
                kq["QHPK_Ma_O_Pho"]  = SEP.join(ds_ma_o_pho)

        except Exception as e:
            kq["Trang_Thai"] = f"Lỗi bóc tách QHPK: {str(e)[:60]}"

    return kq


def lay_thong_tin_quy_hoach(lat, lon):
    """
    Hàm chính: gọi API tại tọa độ gốc.
    Nếu kết quả CHỈ có đất giao thông → thử 8 điểm lân cận (~5.5m)
    để tìm ô đất có chức năng khác.
    """
    cac_diem = [
        (0, 0),
        ( OFFSET_DEGREE,  0),             (-OFFSET_DEGREE,  0),
        (0,               OFFSET_DEGREE), (0,              -OFFSET_DEGREE),
        ( OFFSET_DEGREE,  OFFSET_DEGREE), (-OFFSET_DEGREE, -OFFSET_DEGREE),
        ( OFFSET_DEGREE, -OFFSET_DEGREE), (-OFFSET_DEGREE,  OFFSET_DEGREE),
    ]
    ket_qua_goc = None

    for idx, (d_lat, d_lon) in enumerate(cac_diem):
        if d_lat != 0 or d_lon != 0:
            time.sleep(1.5)

        result_data, loi = _goi_api_mot_diem(float(lat) + d_lat, float(lon) + d_lon)

        # Lỗi nghiêm trọng → dừng
        if loi and any(k in loi for k in ("Bị chặn", "Timeout", "Lỗi mạng")):
            kq = {c: "" for c in CAC_COT_MOI}
            kq["Trang_Thai"] = loi
            return ket_qua_goc or kq

        if loi:
            kq = {c: "" for c in CAC_COT_MOI}
            kq["Trang_Thai"] = loi
            if d_lat == 0 and d_lon == 0:
                ket_qua_goc = kq
            continue

        kq = _boc_tach_ket_qua(result_data)

        # Kiểm tra có ô nào KHÔNG phải giao thông không
        cac_o = [o.strip().lower() for o in kq["QHPK_Chuc_Nang"].split("|")]
        co_o_khac = any("giao thông" not in o for o in cac_o if o)

        if d_lat == 0 and d_lon == 0:
            ket_qua_goc = kq
            if co_o_khac or not kq["QHPK_Chuc_Nang"]:
                return kq  # Điểm gốc đã có ô không phải GT → dùng luôn
            # Toàn giao thông → tiếp tục thử điểm lân cận
        else:
            if co_o_khac:
                kq["Trang_Thai"] = "Thành công (Tự động dịch điểm, điểm gốc chỉ có đất giao thông)"
                return kq

    return ket_qua_goc  # Trả kết quả gốc dù toàn giao thông


# ──────────────────────────────────────────────────────────────
def xu_ly_file_csv(file_dau_vao, file_dau_ra):
    try:
        # Đọc header file input
        with open(file_dau_vao, mode='r', encoding='utf-8-sig') as f:
            cac_cot_goc = csv.DictReader(f).fieldnames

        if not cac_cot_goc or 'Latitude' not in cac_cot_goc or 'Longitude' not in cac_cot_goc:
            print("❌ File CSV đầu vào phải có cột 'Latitude' và 'Longitude'.")
            return

        # Đếm tổng số dòng
        with open(file_dau_vao, mode='r', encoding='utf-8-sig') as f:
            tong_dong = len(list(csv.DictReader(f)))

        toan_bo_cot = cac_cot_goc + CAC_COT_MOI

        print(f"🚀 Bắt đầu xử lý {tong_dong} dòng...\n")
        print(f"   Các ô QHPK gộp bằng dấu ' | ' nếu có nhiều ô.\n")

        with open(file_dau_vao, mode='r', encoding='utf-8-sig') as f_in, \
             open(file_dau_ra,  mode='w', encoding='utf-8-sig', newline='') as f_out:

            reader = csv.DictReader(f_in)
            writer = csv.DictWriter(f_out, fieldnames=toan_bo_cot)
            writer.writeheader()

            for index, row in enumerate(reader, 1):
                link = row.get("Link", "N/A")
                lat  = row.get("Latitude",  "").strip()
                lon  = row.get("Longitude", "").strip()

                print(f"[{index}/{tong_dong}] {link[:45]}... | Tọa độ: {lat}, {lon}")

                if lat and lon:
                    thong_tin = lay_thong_tin_quy_hoach(lat, lon)
                    row.update(thong_tin)

                    # In tóm tắt ra terminal
                    so_o      = thong_tin.get("QHPK_So_O", 0)
                    chuc_nang = thong_tin.get("QHPK_Chuc_Nang", "")
                    ty_le     = thong_tin.get("QHPK_Ty_Le", "")
                    trang_thai = thong_tin.get("Trang_Thai", "")
                    print(f"   ✔ {so_o} ô  |  {chuc_nang}")
                    print(f"   ✔ Tỷ lệ: {ty_le}  |  {trang_thai}")
                else:
                    row["Trang_Thai"] = "Thiếu tọa độ"
                    print(f"   ⚠️  Bỏ qua — thiếu tọa độ")

                writer.writerow(row)
                time.sleep(DELAY_GIUA_DONG)

        print(f"\n✅ Hoàn tất! File kết quả: {file_dau_ra}")
        print(f"\n📋 Ý nghĩa các cột QHPK:")
        print(f"   QHPK_So_O       — Số ô quy hoạch")
        print(f"   QHPK_Chuc_Nang  — Chức năng sử dụng đất từng ô")
        print(f"   QHPK_Dien_Tich  — Diện tích từng ô (m²)")
        print(f"   QHPK_Ty_Le      — % diện tích từng ô trong lô")
        print(f"   QHPK_Ma_QU      — Mã quy ước từng ô")
        print(f"   QHPK_Ma_O_Pho   — Mã ô phố từng ô")
        print(f"   (Các ô phân cách bằng ' | ', theo đúng thứ tự)")

    except FileNotFoundError:
        print(f"❌ Không tìm thấy file '{file_dau_vao}'.")
    except Exception as e:
        print(f"❌ Lỗi nghiêm trọng: {e}")

In [ ]:
if __name__ == "__main__":
    xu_ly_file_csv(FILE_INPUT, FILE_OUTPUT)

🚀 Bắt đầu xử lý 831 dòng...

   Các ô QHPK gộp bằng dấu ' | ' nếu có nhiều ô.

[1/831] https://batdongsan.com.vn/ban-nha-rieng-duong... | Tọa độ: 10.764640163967105, 106.69220041593972
   ✔ 4 ô  |  Đất công trình công cộng (thương mại dịch vụ) | Đất giao thông | Đất giao thông | Đất giao thông
   ✔ Tỷ lệ: 1.85% | 1.29% | 96.19% | 0.67%  |  Thành công
[2/831] https://batdongsan.com.vn/ban-nha-rieng-duong... | Tọa độ: 10.8528630209991, 106.720374186852
   ✔  ô  |  
   ✔ Tỷ lệ:   |  Bị khóa: The land is temporarily locked!
[3/831] https://batdongsan.com.vn/ban-nha-rieng-duong... | Tọa độ: 10.8000280460014, 106.650704397356
   ✔ 3 ô  |  Đất ở - hiện hữu | Đất ở - hiện hữu | Đất giao thông
   ✔ Tỷ lệ: 4.15% | 3.33% | 92.52%  |  Thành công
[4/831] https://batdongsan.com.vn/ban-nha-mat-pho-duo... | Tọa độ: 10.7932583037954, 106.640202947743
   ✔ 3 ô  |  Đất ở - hiện hữu | Đất giao thông | Đất cây xanh công viên - thể dục thể thao
   ✔ Tỷ lệ: 16.93% | 81.01% | 2.06%  |  Thành công
[5/831] http

Tăng hiệu suất bẳng cách bỏ qua những bản ghi không thể sửa.

In [17]:
import csv
import time
import requests
import json
import ast

# ==========================================
# CẤU HÌNH TẠI ĐÂY
# ==========================================
FILE_INPUT       = "/Users/doanlong/Documents/Môn học/Thu thập và Tiền xử lý dữ liệu/house_price_expanded/Bản_ghi_chỉ_có_trong_bds_quyhoach.csv"
FILE_OUTPUT      = "/Users/doanlong/Documents/Môn học/Thu thập và Tiền xử lý dữ liệu/house_price_expanded/Bản_ghi_chỉ_có_trong_bds_quyhoach.csv"
MAX_RETRY        = 3
BACKOFF_FACTOR   = 3
DELAY_GIUA_DONG  = 3
OFFSET_DEGREE    = 0.00005

# Các trạng thái KHÔNG retry dù dữ liệu trống
# (lỗi do bản chất dữ liệu, retry cũng vô ích)
TRANG_THAI_BO_QUA = [
    "server trả về rỗng",
    "bị khóa: the land is temporarily locked!",
]
# ==========================================

session = requests.Session()
session.headers.update({
    "Host":         "sqhkt-qlqh.tphcm.gov.vn",
    "Accept":       "application/json, text/plain, */*",
    "Content-Type": "application/x-www-form-urlencoded",
    "User-Agent":   "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Origin":       "https://thongtinquyhoach.hochiminhcity.gov.vn",
    "Referer":      "https://thongtinquyhoach.hochiminhcity.gov.vn/"
})

SEP = " | "

CAC_COT_QHPK = [
    "Quan_Huyen", "Phuong_Xa", "So_Thua", "So_To", "Dien_Tich_Lo", "Ten_Do_An",
    "QHPK_So_O", "QHPK_Chuc_Nang", "QHPK_Dien_Tich", "QHPK_Ty_Le",
    "QHPK_Ma_QU", "QHPK_Ma_O_Pho", "Trang_Thai",
]


# ──────────────────────────────────────────────────────────────
def can_chay_lai(row):
    """
    Trả về True nếu bản ghi cần chạy lại.
    Tiêu chí: Quan_Huyen trống VÀ Trang_Thai không thuộc danh sách bỏ qua.
    """
    lat = row.get("Latitude", "").strip()
    lon = row.get("Longitude", "").strip()
    if not lat or not lon:
        return False  # Thiếu tọa độ

    quan_huyen = row.get("Quan_Huyen", "").strip()
    if quan_huyen:
        return False  # Đã có dữ liệu

    trang_thai = row.get("Trang_Thai", "").strip().lower()
    la_bo_qua  = any(tt in trang_thai for tt in TRANG_THAI_BO_QUA)
    return not la_bo_qua


# ── Các hàm API (copy từ thu_thap_quy_hoach.py) ───────────────
def _goi_api_mot_diem(lat, lon):
    url  = "https://sqhkt-qlqh.tphcm.gov.vn/computing/930/api/v3.1/a-z/all"
    data = {"Lat": lat, "Lon": lon}

    for attempt in range(1, MAX_RETRY + 1):
        try:
            if attempt > 1:
                time.sleep(BACKOFF_FACTOR * (attempt - 1))

            response = session.post(url, data=data, timeout=15)

            if response.status_code != 200:
                return None, f"Lỗi HTTP {response.status_code}"

            raw_text = response.text.strip() if response.text else ""
            if not raw_text:
                return None, "Server trả về rỗng"

            try:
                result_data = response.json()
            except (json.JSONDecodeError, ValueError):
                try:
                    result_data = ast.literal_eval(raw_text)
                except Exception:
                    return None, f"Lỗi cấu trúc dữ liệu: {raw_text[:50]}"

            if isinstance(result_data, dict) and "error" in result_data:
                return None, f"Bị khóa: {result_data['error']}"
            if result_data.get("blocked") == 1:
                return None, "Bị chặn IP"

            return result_data, None

        except requests.exceptions.Timeout:
            if attempt == MAX_RETRY:
                return None, "Mạng Timeout (Hết lượt thử lại)"
        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRY:
                return None, f"Lỗi mạng: {str(e)[:50]}"
        except Exception as e:
            return None, f"Lỗi không xác định: {str(e)[:50]}"

    return None, "Hết lượt retry"


def _boc_tach_ket_qua(result_data):
    kq = {c: "" for c in CAC_COT_QHPK}
    kq["QHPK_So_O"] = 0
    kq["Trang_Thai"] = "Thành công"

    raw_chung = result_data.get("ThongTinChung", "")
    if raw_chung and raw_chung not in ("[]", "{}"):
        try:
            ttc = json.loads(raw_chung) if isinstance(raw_chung, str) else raw_chung
            if isinstance(ttc, dict):
                kq["Quan_Huyen"]   = ttc.get("tenquanhuyen", "")
                kq["Phuong_Xa"]    = ttc.get("tenphuongxa", "")
                kq["So_Thua"]      = ttc.get("sothua", "")
                kq["So_To"]        = ttc.get("soto", "")
                kq["Dien_Tich_Lo"] = ttc.get("dientich", "")
                dsdoan = ttc.get("dsdoan", [])
                if isinstance(dsdoan, list):
                    kq["Ten_Do_An"] = SEP.join(dsdoan)
        except Exception:
            kq["Trang_Thai"] = "Lỗi bóc tách ThongTinChung"

    raw_qhpk = result_data.get("QHPK", "")
    if raw_qhpk and raw_qhpk not in ("[]", "{}"):
        try:
            qhpk_list = json.loads(raw_qhpk) if isinstance(raw_qhpk, str) else raw_qhpk
            if isinstance(qhpk_list, dict):
                qhpk_list = [qhpk_list]

            if isinstance(qhpk_list, list) and len(qhpk_list) > 0:
                ds_chuc_nang, ds_dien_tich, ds_ty_le, ds_ma_qu, ds_ma_o_pho = [], [], [], [], []

                for item in qhpk_list:
                    props     = item.get("properties", {}) if isinstance(item, dict) else {}
                    chuc_nang = props.get("chucnang") or props.get("chucNang") or ""
                    dien_tich = props.get("dientich", "")
                    ty_le     = props.get("tldientich", "")
                    ma_qu     = props.get("maquyuoc", "") or ""
                    ma_o_pho  = props.get("maopho", "")   or ""

                    try:
                        dien_tich = f"{float(dien_tich):.2f}"
                    except (TypeError, ValueError):
                        dien_tich = str(dien_tich)
                    try:
                        ty_le = f"{float(ty_le):.2f}%"
                    except (TypeError, ValueError):
                        ty_le = str(ty_le)

                    ds_chuc_nang.append(chuc_nang)
                    ds_dien_tich.append(dien_tich)
                    ds_ty_le.append(ty_le)
                    ds_ma_qu.append(str(ma_qu))
                    ds_ma_o_pho.append(str(ma_o_pho))

                kq["QHPK_So_O"]      = len(qhpk_list)
                kq["QHPK_Chuc_Nang"] = SEP.join(ds_chuc_nang)
                kq["QHPK_Dien_Tich"] = SEP.join(ds_dien_tich)
                kq["QHPK_Ty_Le"]     = SEP.join(ds_ty_le)
                kq["QHPK_Ma_QU"]     = SEP.join(ds_ma_qu)
                kq["QHPK_Ma_O_Pho"]  = SEP.join(ds_ma_o_pho)
        except Exception as e:
            kq["Trang_Thai"] = f"Lỗi bóc tách QHPK: {str(e)[:60]}"

    return kq


def lay_thong_tin_quy_hoach(lat, lon):
    cac_diem = [
        (0, 0),
        ( OFFSET_DEGREE,  0),             (-OFFSET_DEGREE,  0),
        (0,               OFFSET_DEGREE), (0,              -OFFSET_DEGREE),
        ( OFFSET_DEGREE,  OFFSET_DEGREE), (-OFFSET_DEGREE, -OFFSET_DEGREE),
        ( OFFSET_DEGREE, -OFFSET_DEGREE), (-OFFSET_DEGREE,  OFFSET_DEGREE),
    ]
    ket_qua_goc = None

    for idx, (d_lat, d_lon) in enumerate(cac_diem):
        if d_lat != 0 or d_lon != 0:
            time.sleep(1.5)

        result_data, loi = _goi_api_mot_diem(float(lat) + d_lat, float(lon) + d_lon)

        if loi and any(k in loi for k in ("Bị chặn", "Timeout", "Lỗi mạng")):
            kq = {c: "" for c in CAC_COT_QHPK}
            kq["Trang_Thai"] = loi
            return ket_qua_goc or kq

        if loi:
            kq = {c: "" for c in CAC_COT_QHPK}
            kq["Trang_Thai"] = loi
            if d_lat == 0 and d_lon == 0:
                ket_qua_goc = kq
            continue

        kq = _boc_tach_ket_qua(result_data)
        cac_o = [o.strip().lower() for o in kq["QHPK_Chuc_Nang"].split("|")]
        co_o_khac = any("giao thông" not in o for o in cac_o if o)

        if d_lat == 0 and d_lon == 0:
            ket_qua_goc = kq
            if co_o_khac or not kq["QHPK_Chuc_Nang"]:
                return kq
        else:
            if co_o_khac:
                kq["Trang_Thai"] = "Thành công (Tự động dịch điểm, điểm gốc chỉ có đất giao thông)"
                return kq

    return ket_qua_goc


# ──────────────────────────────────────────────────────────────
def chay_lai_ban_ghi_loi(file_dau_vao, file_dau_ra):
    try:
        # Đọc toàn bộ file vào RAM
        with open(file_dau_vao, mode='r', encoding='utf-8-sig') as f:
            reader   = csv.DictReader(f)
            fieldnames = reader.fieldnames
            tat_ca_dong = list(reader)

        if not fieldnames:
            print("❌ File trống hoặc không đọc được header.")
            return

        # Xác định các dòng cần chạy lại
        chi_so_can_retry = [
            i for i, row in enumerate(tat_ca_dong) if can_chay_lai(row)
        ]

        tong_dong    = len(tat_ca_dong)
        tong_retry   = len(chi_so_can_retry)
        tong_ok      = tong_dong - tong_retry

        print(f"📂 Tổng số bản ghi   : {tong_dong}")
        print(f"✅ Đã có dữ liệu     : {tong_ok}")
        print(f"🔁 Cần chạy lại      : {tong_retry}")

        if tong_retry == 0:
            print("\n🎉 Không có bản ghi nào cần chạy lại. File đã đầy đủ!")
            return

        print(f"\n🚀 Bắt đầu retry {tong_retry} bản ghi...\n")

        thanh_cong = 0
        van_loi    = 0

        for thu_tu, i in enumerate(chi_so_can_retry, 1):
            row  = tat_ca_dong[i]
            lat  = row.get("Latitude",  "").strip()
            lon  = row.get("Longitude", "").strip()
            link = row.get("Link", "N/A")

            trang_thai_cu = row.get("Trang_Thai", "").strip()
            print(f"[{thu_tu}/{tong_retry}] Dòng #{i+1} | {link[:40]}...")
            print(f"   Lỗi cũ : {trang_thai_cu}")

            thong_tin = lay_thong_tin_quy_hoach(lat, lon)
            tat_ca_dong[i].update(thong_tin)

            trang_thai_moi = thong_tin.get("Trang_Thai", "")
            so_o      = thong_tin.get("QHPK_So_O", 0)
            chuc_nang = thong_tin.get("QHPK_Chuc_Nang", "")

            if thong_tin.get("Quan_Huyen", "").strip():
                thanh_cong += 1
                print(f"   ✔ Thành công | {so_o} ô | {chuc_nang}")
            else:
                van_loi += 1
                print(f"   ✘ Vẫn lỗi   | {trang_thai_moi}")

            time.sleep(DELAY_GIUA_DONG)

        # Ghi lại toàn bộ file (kể cả dòng không retry, giữ nguyên)
        with open(file_dau_ra, mode='w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(tat_ca_dong)

        print(f"\n{'='*50}")
        print(f"✅ Retry hoàn tất! File đã lưu: {file_dau_ra}")
        print(f"   Thành công thêm : {thanh_cong} bản ghi")
        print(f"   Vẫn còn lỗi    : {van_loi} bản ghi")
        if van_loi > 0:
            print(f"   → Chạy lại script này thêm lần nữa để retry tiếp.")

    except FileNotFoundError:
        print(f"❌ Không tìm thấy file '{file_dau_vao}'.")
    except Exception as e:
        print(f"❌ Lỗi nghiêm trọng: {e}")


if __name__ == "__main__":
    chay_lai_ban_ghi_loi(FILE_INPUT, FILE_OUTPUT)

📂 Tổng số bản ghi   : 831
✅ Đã có dữ liệu     : 831
🔁 Cần chạy lại      : 0

🎉 Không có bản ghi nào cần chạy lại. File đã đầy đủ!
